# MathLive input widgets showcase

This notebook explains the **figure-free semantic MathLive layer**. The goal is to show how the toolkit turns user-facing math input into canonical identifiers and SymPy expressions without relying on `Figure` or plot editors.

What you should learn here:

- what an `ExpressionContext` registers
- how `IdentifierInput` and `ExpressionInput` use that context
- why MathJSON is the preferred transport format
- which errors are intentional safeguards rather than missing features

## Setup

The notebook adds to `sys.path` the repository `src/` folder with respect to the first parent that contains the file `.root` `. There is no package installation step and no figure-related setup.

In [ ]:
import import_setup

In [ ]:
import sympy as sp
from IPython.display import display, Markdown, Latex

from gu_toolkit import NamedFunction
from gu_toolkit.identifiers import symbol
from gu_toolkit.mathlive import (
    ExpressionContext,
    ExpressionInput,
    IdentifierInput,
    MathJSONParseError,
    mathjson_to_identifier,
)

## 0. Semantic-math helpers - Documentation


The semantic-math helpers are meant to be discovered from `help(...)`, so their docstrings need to explain more than the Python signature. In particular, the public docs now call out the difference between **canonical names** (for example `theta__x`) and the **display LaTeX** that the notebook shows to readers.

Good places to inspect while reading this notebook are `help(symbol)`, `help(ExpressionContext)`, and `help(IdentifierInput.parse_value)`. Those docstrings now point back to the semantic-math guide, the API-discovery guide, and the focused regression tests.


In [ ]:
symbol?

In [ ]:
ExpressionContext?

In [ ]:
# This showcase note book is BAD. Organize it from top to bottom not from bottom up. 
# It is not necessary to know the internals first. First you should give an example of how to get a working input widget with restrictions on which symbols are allowed
# You should provide a markdown cell with instructions on how to manually test the input widget. I tried manually and could not figure it out. Value was aways returning 0. 
# DO NOT set the json programmatically in your example. Actually rely on the USER doing it. Then provide another cell in which you do it by injecting javascript.
# Doing it programmatically shows nothing
# Also, move this ExpressionContext part further down. Only for users that need to understand internals. The top level should be SHOWCASING functionality, not building up architecture. 
# Completely revisit your approach from here down. 

## 1. Build an explicit semantic context

`ExpressionContext` lists which names are symbols, which are functions, and how they should render. It is the contract between the backend and the widgets. 

The backend stores **canonical names** such as `velocity`, `theta__x`, and `Force_t`. Display LaTeX is derived from those names instead of being stored as the identity itself.

### A note on `symbol(...)` and SymPy

The toolkit `symbol(...)` helper intentionally returns a regular SymPy `Symbol`, but it first validates the canonical identifier and optionally records a display-LaTeX override. That keeps the semantic-math layer interoperable with SymPy while still enforcing the toolkit's identifier rules.


In [ ]:
# Build a context with both symbols and callable names.
@NamedFunction
def Force(x):
    return x

velocity = symbol('velocity')
theta_x_literal = symbol('theta__x')
a_12 = symbol('a_1_2')
x = symbol('x')

ctx = ExpressionContext.from_symbols(
    [velocity, theta_x_literal, a_12, x],
    functions=[Force, 'Force_t', 'Force__x'],
    include_named_functions=False,
)

# Export the frontend-facing manifest used by MathLive widgets.
manifest = ctx.transport_manifest(field_role='expression')
symbols_by_name = {entry['name']: entry for entry in manifest['symbols']}
functions_by_name = {entry['name']: entry for entry in manifest['functions']}

assert symbols_by_name['velocity']['latex'] == r'\mathrm{velocity}'
assert symbols_by_name['theta__x']['latex'] == r'\mathrm{theta\_x}'
assert functions_by_name['Force']['latexHead'] == r'\operatorname{Force}'
assert functions_by_name['Force_t']['template'] == r'\operatorname{Force}_{t}(#0)'

# Pull out a smaller summary so the tutorial stays readable.
summary = {
    'fieldRole': manifest['fieldRole'],
    'symbols': [
        {key: symbols_by_name[name][key] for key in ('name', 'latex', 'atoms')}
        for name in ('velocity', 'theta__x', 'a_1_2')
    ],
    'functions': [
        {key: functions_by_name[name][key] for key in ('name', 'latexHead', 'template')}
        for name in ('Force', 'Force_t', 'Force__x')
    ],
}
summary


## 2. `IdentifierInput` normalizes what the user typed

The widget can receive either plain LaTeX text or MathJSON. In both cases, the backend should reduce the result to the same canonical identifier.

This cell also doubles as an integration check: the asserts make sure the documented examples keep working over time.

In [ ]:
# BUG: DO NOT MAKE THE SAME cell double as an integration cell. STOP DOING THAT!!!1
# For integration cells make them explicilty only integration cells, make them go AFTER manual tests, 
# make them use javascript injection to simulate user input.
# Also, for integration cells, test ALL functinoality and print out a report. A user will see patterns and understand what is going on. 

#BUG for manual examples write in markdown what the user should do and what to expect as output. Decide whether output necessitates rerunning a cell later to get output
# or even better: provide a dynamic OUTPUT field Output() in which you display() the result. 

# The widget accepts either LaTeX text or MathJSON and normalizes both.
identifier_widget = IdentifierInput(context=ctx, value=r'a_{1,2}')
display(identifier_widget)

assert identifier_widget.parse_value() == 'a_1_2'

identifier_widget.value = ''
identifier_widget.math_json = ['Subscript', 'a', ['Tuple', 1, 2]]
assert identifier_widget.parse_value() == 'a_1_2'

display(Markdown('`IdentifierInput` normalizes both LaTeX and MathJSON to the same canonical name.'))


In [ ]:
identifier_widget.get_state()

## 3. `ExpressionInput` keeps known names atomic

Without a context, names like `velocity` could be parsed as a product of single-letter symbols. With a context, the expression widget keeps registered names atomic and keeps registered functions callable.

In [ ]:
expression_widget = ExpressionInput(
    context=ctx,
    value=r'\operatorname{Force}(\mathrm{theta\_x}) + 2\mathrm{velocity}',
)
display(expression_widget)

parsed_from_text = expression_widget.parse_value()
assert parsed_from_text == Force(theta_x_literal) + 2 * velocity
assert r'\operatorname{Force}' in ctx.render_latex(parsed_from_text)
assert r'\mathrm{theta\_x}' in ctx.render_latex(parsed_from_text)
#BUG Huge bug that contradicts my instructions directly. LaTeX rendering should be done BY SYMPY and its machinery using the optional latex represenation field. Completely abolish any custom render_latex function from the code.
parsed_from_text



## 4. MathJSON is the preferred transport

When the MathLive Compute Engine is available, the widget can send a structured MathJSON payload instead of relying only on string parsing. This avoids many classes of ambiguity and gives the backend a cleaner, more testable boundary.

In [ ]:
expression_widget.value = ''
expression_widget.math_json = [
    'Add',
    ['Force', 'theta__x'],
    ['Multiply', 'velocity', 2],
]

parsed_from_mathjson = expression_widget.parse_value()
assert parsed_from_mathjson == Force(theta_x_literal) + 2 * velocity
assert parsed_from_mathjson == parsed_from_text

parsed_from_mathjson

## 5. Some errors are intentional safeguards

A transport spelling such as `theta_x` is ambiguous when no context says whether it should mean a real subscript or a literal underscore inside the atom. In that situation the toolkit **raises** instead of guessing.

This is an example of the general design philosophy: fail early at the semantic boundary, not later inside unrelated plotting code.

In [ ]:
try:
    mathjson_to_identifier('theta_x')
except MathJSONParseError as exc:
    ambiguous_message = str(exc)
else:
    raise AssertionError('Expected an ambiguity error for an unregistered single-underscore name.')

assert 'ambiguous' in ambiguous_message.lower()
assert mathjson_to_identifier('theta__x', context=ctx) == 'theta__x'

ambiguous_message

## Try your own variations

Good experiments after reading this notebook:

- register a new symbol such as `sigma_1` or `sigma__x` and compare the rendered LaTeX
- add a new function name such as `Energy_t` and inspect the transport manifest again
- change the widget text to plain canonical input (`Force(theta__x) + 2*velocity`) and compare it with the explicit LaTeX form
- compare what happens with `theta_x` before and after registering it in the context